# Step 5 — DINOv3 ConvNeXt-Tiny

DINOv3 is the pretraining method; ConvNeXt-Tiny is the backbone architecture. We retain the pretrained backbone, modify its RGB input stem for five satellite bands, and attach the shared U-Net decoder used by the other backbones.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/hriship618/coastline-image-segmentation.git"
repository = Path("/content/coastline-image-segmentation")
if not repository.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repository)], check=True)
os.chdir(repository)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[ml]"], check=True)
source_directory = str(repository / "src")
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)
print(f"Working from: {repository}")

## 1. Authenticate with Hugging Face
The DINOv3 weights are gated. Request access on the model page, add `HF_TOKEN` under Colab's Secrets panel, and grant this notebook access. The token is read without printing it.

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("HF token loaded without displaying it.")

## 2. Create the five-band batch
The channel order is RGB, NIR, SWIR. Keeping RGB first matters because the first three pretrained channel weights are copied without reordering.

In [ ]:
import torch
from coastlearn.synthetic import make_synthetic_coast

examples = [make_synthetic_coast(height=128, width=128, seed=seed) for seed in (7, 11)]
images = torch.stack([torch.from_numpy(image) for image, _ in examples])
masks = torch.stack([torch.from_numpy(mask) for _, mask in examples])
print("images:", images.shape, "# RGB + NIR + SWIR")

## 3. Load DINOv3 and adapt its input
Loading the model downloads the pretrained weights into the Colab runtime. The original stem accepts three channels. Our constructor replaces it with a five-channel convolution.

In [ ]:
from coastlearn.models import build_dinov3_convnext_tiny

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_dinov3_convnext_tiny(in_channels=5, num_classes=2).to(device)
images = images.to(device)
masks = masks.to(device)
print("stem input channels:", model.input_stem.in_channels)
print("stem weight shape:  ", tuple(model.input_stem.weight.shape))

## 4. Verify extra-channel initialization
The NIR and SWIR filters initially equal the mean RGB filter. They are merely sensible starting values; training will allow them to specialize.

In [ ]:
weights = model.input_stem.weight.detach()
mean_rgb = weights[:, :3].mean(dim=1)
print("NIR starts as mean RGB: ", torch.allclose(weights[:, 3], mean_rgb))
print("SWIR starts as mean RGB:", torch.allclose(weights[:, 4], mean_rgb))

## 5. Inspect features and output
The shared decoder combines all four feature resolutions through skip connections before producing the land/water mask.

In [ ]:
with torch.no_grad():
    feature_pyramid = model.extract_features(images)
    logits = model(images)
for index, features in enumerate(feature_pyramid, start=1):
    print(f"stage {index}: {tuple(features.shape)}")
print("land/water logits:", tuple(logits.shape))

## 6. Perform one update
The complete DINOv3 backbone, including its adapted input stem, receives the smaller learning rate. The newly created segmentation head receives the larger learning rate.

In [ ]:
from coastlearn.training import build_cross_entropy_loss, build_finetuning_optimizer, train_one_batch

loss_function = build_cross_entropy_loss(ignore_index=255).to(device)
optimizer = build_finetuning_optimizer(
    model, backbone_learning_rate=1e-5, head_learning_rate=1e-3
)
result = train_one_batch(model, images, masks, optimizer, loss_function)
print(result)

## What we are testing

Ordinary ConvNeXt and DINOv3 ConvNeXt have the same backbone structure and the same shared decoder. Their important experimental difference is pretraining. ResNet uses the same decoder design and widths, while its encoder channel counts necessarily differ.